In [66]:
import pandas as pd
import json


In [67]:

centerline_json = "./data/raw/centerlines/Jefferson_County_KY_Street_Centerlines.geojson"

def get_records(intersection_data_path):
    with open(intersection_data_path, 'r') as file:
        data = json.load(file)
    features = data['features']
    for feature in features:
        properties = feature['properties']
        geometry = feature['geometry']
        # properties['geo_type'] = geometry['type'] # Always == "Point". Not useful.
        properties['GEOMETRY'] = geometry['coordinates']
        yield properties

centerlines_raw = pd.DataFrame.from_records(get_records(centerline_json))
centerlines_raw.head()

,OBJECTID,RWCOMPKEY,RWCOMPKEYPARENT,LOW_ADDRESS,FROM_ADDRESS,TO_ADDRESS,LFROM,LTO,RFROM,RTO,...,ALIAS3_STRNAME,ALIAS3_TYPE,ALIAS3_SIFID,ALIAS4_DIR,ALIAS4_STRNAME,ALIAS4_TYPE,ALIAS4_SIFID,GLOBALID,SHAPELEN,GEOMETRY
0,1,24074,0,6300,6300,6399,6301,6399,6300,6398,...,,,0,,,,0,{496EE316-2749-408F-AD2D-16DD49C99017},325.140306,"[[-85.6809503218278, 38.158867088749105], [-85..."
1,2,20311,0,1600,1600,1619,1601,1619,1600,1618,...,,,0,,,,0,{AA2CAA18-473D-416D-8973-C74538F2BB31},447.052294,"[[-85.80120122370631, 38.23056379303306], [-85..."
2,3,21109,0,1700,1700,1705,1701,1705,1700,1704,...,,,0,,,,0,{A9B57033-3F9C-4CA7-8ED3-0F17D5DF6C14},552.489593,"[[-85.80501498815028, 38.228933021550915], [-8..."
3,4,20926,0,505,505,519,505,519,506,518,...,,,0,,,,0,{F40EEA69-C8E4-4A2B-983D-670A93C87BFE},193.929089,"[[-85.68020545343364, 38.24806713661984], [-85..."
4,5,19036,0,730,730,799,731,799,730,798,...,,,0,,,,0,{D0515DFE-8CF3-450F-9470-A1C35303ACA7},125.862060,"[[-85.74203979766622, 38.211814770925905], [-8..."


In [68]:
INDEX = 'OBJECTID'

SIF_INFO = ['SIFID', 'SIFCODE', 'SIFIDLOW', 'LOCROSSSIF', 'SIFIDHI', 'HICROSSSIF']

# name info helpful for presentation but no so much for searching road network
# probably keep on of these like ROADNAME in working data for sanity checks but most of
# this can be looked up on an as needed basis. 
# Move into a separate linked database. 
NAME_INFO = ['ROADNAME']

NAME_INFO_TO_DROP = ['AVDIR', 'AVPRETYPE', 'AVSTNAME', 'AVSTTYPE', 'AVSUFDIR', 
                     'DIR', 'STRNAME', 'TYPE', 'ROUTE_LBL']

# can always look this up other ways (probably)
CROSS_STREET_NAMES = ['LOCROSSPR', 'LOCROSSNA', 'LOCROSSSU','HICROSSPR', 'HICROSSNA', 'HICROSSSU']


# Use this to filter out stuff like interstates
ROAD_CLASS = ['CORE_CLASS']
ROAD_CLASS_REDUNDANT = ['CLASS_ABRV', 'STCL_CLASS']


# seems like it should be helpful but I can't see how exactly.
GEOMETRY_INFO = ['LENGTH', 'SHAPELEN']


# find out the structure here?
MIDAS = ['COMPKEY', 'COMPTYPE', 'UNITID', 'UNITID2', 'RWCOMPKEY', 'RWCOMPKEYPARENT']

# what do these mean?
ALIASES = ['ALIAS1_DIR', 'ALIAS1_STRNAME', 'ALIAS1_TYPE', 'ALIAS1_SIFID',
           'ALIAS2_DIR', 'ALIAS2_STRNAME', 'ALIAS2_TYPE', 'ALIAS2_SIFID',
           'ALIAS3_DIR', 'ALIAS3_STRNAME', 'ALIAS3_TYPE', 'ALIAS3_SIFID',
           'ALIAS4_DIR', 'ALIAS4_STRNAME', 'ALIAS4_TYPE', 'ALIAS4_SIFID']

# not helpful to me
ADDRESS_INFO = ['LOW_ADDRESS','FROM_ADDRESS', 'TO_ADDRESS', 
                'LFROM', 'LTO', 'RFROM', 'RTO', 'ZIP_LEFT', 'ZIP_RIGHT']
# no need to look up house addresses

SPEED_INFO = ['FTIMP', 'TFIMP', 'SPEED'] 
# don't care about speed

CITY_USE = ['MUNI_NAME', 'COUNDIST', 'MAIN_DIST', 'OWNER_NAME', 'OWNER_ABRV',
             'MAIN_RESP', 'MAIN_ABRV', 'ONE_WAY', 'MAPNO', 'GLOBALID']
# dont care about city maintenance districts/owners or what have you
# don't care about one ways
# don't know what globalid is but it seems to to with LOJIC internally. Same with MAPNO

# simply not useful to me:
DATABASE_META = ['ENTERED_BY', 'ENTERED_ON', 'UPDATED_BY', 'UPDATED_ON']


In [69]:
columns_to_drop = [
    # midas
    'COMPKEY', 'COMPTYPE', 'UNITID', 'UNITID2', 'RWCOMPKEY', 'RWCOMPKEYPARENT',
    
    # no need to look up house addresses
    'LOW_ADDRESS','FROM_ADDRESS', 'TO_ADDRESS', 'LFROM', 'LTO', 'RFROM', 'RTO',
      'ZIP_LEFT', 'ZIP_RIGHT',

    'AVDIR', 'AVPRETYPE', 'AVSTNAME', 'AVSTTYPE', 'AVSUFDIR', 'DIR', 'STRNAME',
     'TYPE', 'ROUTE_LBL',

    # 
    'LENGTH', 'SHAPELEN',

    # redundant information
    'CLASS_ABRV', 'STCL_CLASS',

    # don't care about speed info
    'FTIMP', 'TFIMP', 'SPEED',

    'MUNI_NAME', 'COUNDIST', 'MAIN_DIST', 'OWNER_NAME', 'OWNER_ABRV', 'MAIN_RESP', 'MAIN_ABRV',
     
    'ONE_WAY', 'MAPNO', 'GLOBALID',

    'ENTERED_BY', 'ENTERED_ON', 'UPDATED_BY', 'UPDATED_ON',
    ]

# what do these mean?
ALIASES = ['ALIAS1_DIR', 'ALIAS1_STRNAME', 'ALIAS1_TYPE', 'ALIAS1_SIFID',
           'ALIAS2_DIR', 'ALIAS2_STRNAME', 'ALIAS2_TYPE', 'ALIAS2_SIFID',
           'ALIAS3_DIR', 'ALIAS3_STRNAME', 'ALIAS3_TYPE', 'ALIAS3_SIFID',
           'ALIAS4_DIR', 'ALIAS4_STRNAME', 'ALIAS4_TYPE', 'ALIAS4_SIFID']


In [70]:
#df = df.drop(MIDAS, axis=1)
df1 = centerlines_raw.drop(columns_to_drop, axis=1)
df1 = df1.drop(ALIASES, axis=1)

def compress_cross_street_names(df):
    low_cross_columns = ["LOCROSSPR", "LOCROSSNA", "LOCROSSSU"]
    low_cross_names = df[low_cross_columns]
    df["low_cross_ROADNAME"] = low_cross_names.apply(" ".join, axis=1).str.strip()
    df = df.drop(low_cross_columns, axis=1)

    hi_cross_columns = ['HICROSSPR', 'HICROSSNA', 'HICROSSSU']
    hi_cross_names = df[hi_cross_columns]
    df["hi_cross_ROADNAME"] = hi_cross_names.apply(" ".join, axis=1).str.strip()
    return df.drop(hi_cross_columns, axis=1)

df2 = compress_cross_street_names(df1)
df2.head()

,OBJECTID,SIFID,SIFCODE,ROADNAME,SIFIDLOW,LOCROSSSIF,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOMETRY,low_cross_ROADNAME,hi_cross_ROADNAME
0,1,8665,9887,SERENITY CT,1550,1557,8594,9811,LOCAL,"[[-85.6809503218278, 38.158867088749105], [-85...",DELLAFAY DR,DEAD END
1,2,5926,6570,S 28TH ST,2854,2934,2470,2499,LOCAL,"[[-85.80120122370631, 38.23056379303306], [-85...",W HILL ST,W GAULBERT AVE
2,3,473,0458,BEECH ST,6487,7212,10596,D596,LOCAL,"[[-85.80501498815028, 38.228933021550915], [-8...",WILSON AVE,DR WILLIAM G WEATHERS DR
3,4,2442,2470,GARDEN DR,13394,5391,4702,5128,PRIMARY COLLECTOR,"[[-85.68020545343364, 38.24806713661984], [-85...",RAINBOW DR,POPPY WAY
4,5,4573,4974,PARKWAY DR,4162,4476,8594,9811,LOCAL,"[[-85.74203979766622, 38.211814770925905], [-8...",MOUNT CLAIRE AVE,DEAD END


In [71]:
def create_clean_df(df):
    df = df.drop(columns_to_drop, axis=1)
    df = df.drop(ALIASES, axis=1)
    
    df = compress_cross_street_names(df)
    
    return df[['OBJECTID', 'ROADNAME', 'SIFID', 'SIFCODE', 
               'low_cross_ROADNAME', 'SIFIDLOW', 'LOCROSSSIF',
               'hi_cross_ROADNAME','SIFIDHI', 'HICROSSSIF',
               'CORE_CLASS', 'GEOMETRY']]

centerlines_clean = create_clean_df(centerlines_raw)
centerlines_clean.head()

,OBJECTID,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOMETRY
0,1,SERENITY CT,8665,9887,DELLAFAY DR,1550,1557,DEAD END,8594,9811,LOCAL,"[[-85.6809503218278, 38.158867088749105], [-85..."
1,2,S 28TH ST,5926,6570,W HILL ST,2854,2934,W GAULBERT AVE,2470,2499,LOCAL,"[[-85.80120122370631, 38.23056379303306], [-85..."
2,3,BEECH ST,473,0458,WILSON AVE,6487,7212,DR WILLIAM G WEATHERS DR,10596,D596,LOCAL,"[[-85.80501498815028, 38.228933021550915], [-8..."
3,4,GARDEN DR,2442,2470,RAINBOW DR,13394,5391,POPPY WAY,4702,5128,PRIMARY COLLECTOR,"[[-85.68020545343364, 38.24806713661984], [-85..."
4,5,PARKWAY DR,4573,4974,MOUNT CLAIRE AVE,4162,4476,DEAD END,8594,9811,LOCAL,"[[-85.74203979766622, 38.211814770925905], [-8..."


In [74]:
# write cleaner files for future use
save_location = "./data/cleaner/centerlines_clean.json"

centerlines_clean.to_json(save_location, orient='records', lines=True, double_precision=15)


In [73]:
def read_in_centerlines(path_to_data):
    df = pd.read_json(path_to_data, orient='records', lines=True)
    return df.set_index("OBJECTID")

read_in_centerlines(save_location)

,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOMETRY
OBJECTID,,,,,,,,,,,
1,SERENITY CT,8665,9887,DELLAFAY DR,1550,1557,DEAD END,8594,9811,LOCAL,"[[-85.6809503218278, 38.158867088749105], [-85..."
2,S 28TH ST,5926,6570,W HILL ST,2854,2934,W GAULBERT AVE,2470,2499,LOCAL,"[[-85.80120122370631, 38.23056379303306], [-85..."
3,BEECH ST,473,0458,WILSON AVE,6487,7212,DR WILLIAM G WEATHERS DR,10596,D596,LOCAL,"[[-85.80501498815028, 38.228933021550915], [-8..."
4,GARDEN DR,2442,2470,RAINBOW DR,13394,5391,POPPY WAY,4702,5128,PRIMARY COLLECTOR,"[[-85.68020545343364, 38.24806713661984], [-85..."
5,PARKWAY DR,4573,4974,MOUNT CLAIRE AVE,4162,4476,DEAD END,8594,9811,LOCAL,"[[-85.74203979766622, 38.211814770925905], [-8..."
...,...,...,...,...,...,...,...,...,...,...,...
180232,BROOKE ELIZABETH WAY,15329,F540,DEAD END,8594,9811,LOGISTICS AIRPARK DR,15331,F541,LOCAL,"[[-85.7085947306709, 38.143298896060436], [-85..."
180233,AIKEN RIDGE DR,15611,F760,AIKEN RIDGE CIR,15610,F759,ASHER CT,15642,F791,LOCAL,"[[-85.45093405413414, 38.27043985542058], [-85..."
180234,AIKEN RIDGE DR,15611,F760,ASHER CT,15642,F791,AIKEN RIDGE CIR,15610,F759,LOCAL,"[[-85.45174798320075, 38.270920967505994], [-8..."
